## Problem: Add Grad-CAM to Explain Model Predictions

### Problem Statement
Use Grad-CAM (Gradient-weighted Class Activation Mapping, Selvaraju et al., 2017) to explain the predictions of a convolutional neural network. Grad-CAM highlights the regions of the input image that contribute most to the score of a chosen class.

For a target class $c$ with pre-softmax score (logit) $y^c$, and the $K$ feature maps $A^k$ (each $h \times w$) of a late convolutional layer:
$$
\alpha_k^c = \frac{1}{Z} \sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}, \qquad
L^c_\text{Grad-CAM} = \operatorname{ReLU}\Big(\sum_k \alpha_k^c A^k\Big), \qquad Z = h \cdot w
$$
The channel weights $\alpha_k^c$ are the global-average-pooled gradients; the ReLU keeps only regions with a *positive* influence on class $c$. The map has the layer's spatial size (7×7 for ResNet-18's last block), so it is upsampled to the image size for display.

### Requirements

Implement two functions. They are checked on tiny random CNNs, so nothing is downloaded for grading.

1. **`grad_cam(model, target_layer, x, target_class=None)`** → the raw Grad-CAM maps, shape `(N, h, w)`.
   - `x` is a batch `(N, C, H, W)`; `target_layer` is a submodule of `model` whose output is the feature maps $A$, shape `(N, K, h, w)`; `model(x)` returns logits `(N, num_classes)`.
   - Capture $A$ with a **forward hook** on `target_layer`, and get $\partial y^c / \partial A$ from the backward pass — a hook on the tensor $A$ (`A.register_hook`) plus `backward()`, or `torch.autograd.grad`.
   - `target_class`: `None` → each sample's predicted class (its highest logit); an `int` → that class for every sample; a `(N,)` LongTensor → one class per sample.
   - Backpropagate the class **logit**, not the softmax probability (which mixes in every other class). Each sample gets its **own** weights $\alpha^c$: average the gradients over the spatial dimensions only, never over the batch.
   - Return $\operatorname{ReLU}(\sum_k \alpha_k^c A^k)$ at the layer's resolution, **not normalised**.
   - **Remove every hook you registered** before returning, also when an error is raised (`try`/`finally`). A forgotten hook keeps running, and storing tensors, on every later forward pass.
   - It must work on a **frozen** model (every parameter has `requires_grad=False`, as for a pretrained backbone) called inside `torch.no_grad()`, without changing the parameters' `requires_grad` flags or switching gradients on globally.

2. **`upsample_cam(cam, size)`** → `(N, H, W)` heatmaps for display:
   - upsample `cam` `(N, h, w)` to `size = (H, W)` with `F.interpolate(..., mode="bilinear", align_corners=False)`;
   - divide each map by **its own** maximum, so every heatmap lies in $[0, 1]$; an all-zero map must stay zero, not NaN (add a small eps such as `1e-8` to the maximum).

3. **Demo** (provided): explain a pretrained ResNet-18's prediction on a photo with `target_layer = model.layer4[-1]`, the last residual block (the last spatial feature maps before global pooling), and overlay the heatmap on the image. This part downloads the weights and the image.

### Notes
- The old `module.register_backward_hook` is deprecated and can report the wrong tensors; a hook on the output tensor (or `torch.autograd.grad`) is the robust way to get $\partial y^c / \partial A$.
- When $A$ feeds global average pooling and a linear layer directly (as in ResNet), $\partial y^c / \partial A^k_{ij} = W_{ck} / Z$ is constant over space, and Grad-CAM reduces to CAM (Zhou et al., 2016): $\operatorname{ReLU}(\sum_k W_{ck} A^k) / Z$. The self-check below uses that.

In [ ]:
# Explain a CNN model's predictions using Grad-CAM in PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def grad_cam(model, target_layer, x, target_class=None):
    """
    Grad-CAM maps of `target_layer` for a batch.

    Args:
        model:        nn.Module mapping (N, C, H, W) images to (N, num_classes) logits.
        target_layer: submodule of `model` whose output is the feature maps A: (N, K, h, w).
        x:            (N, C, H, W) input batch.
        target_class: None (each sample's argmax), an int (same class for all samples),
                      or a (N,) LongTensor (one class per sample).

    Returns:
        (N, h, w) tensor: ReLU(sum_k alpha_k A^k), alpha_k = spatial mean of d logit_c / d A^k.
        Raw, not normalised. All hooks are removed before returning.
    """
    store = {}

    def keep_activations(module, inputs, output):
        store["A"] = output                      # A: (N, K, h, w), still part of the graph

    handle = target_layer.register_forward_hook(keep_activations)
    try:
        # enable_grad + an input that requires grad: a graph is built even inside
        # torch.no_grad() and for a frozen model, without touching the parameters.
        with torch.enable_grad():
            logits = model(x.detach().requires_grad_(True))            # (N, num_classes)
            if target_class is None:
                target_class = logits.argmax(dim=1)
            cls = torch.as_tensor(target_class, device=logits.device).long().reshape(-1)
            cls = cls.expand(logits.shape[0])                          # int -> one per sample
            # Sum of each sample's own class logit: samples do not interact, so one
            # backward pass gives every sample the gradient of its own score.
            score = logits.gather(1, cls[:, None]).sum()
            A = store["A"]
            (grads,) = torch.autograd.grad(score, A)                  # d y^c / d A, like A
    finally:
        handle.remove()                          # also when something above raised

    weights = grads.mean(dim=(2, 3), keepdim=True)                   # alpha: (N, K, 1, 1)
    return torch.relu((weights * A).sum(dim=1)).detach()             # (N, h, w)


def upsample_cam(cam, size):
    """
    Args:
        cam:  (N, h, w) raw Grad-CAM maps.
        size: (H, W) output size.

    Returns:
        (N, H, W): bilinear upsampling (align_corners=False), each map divided by its own
        maximum (+ a small eps, so an all-zero map stays zero).
    """
    up = F.interpolate(cam.unsqueeze(1), size=size, mode="bilinear", align_corners=False)
    up = up.squeeze(1)                                                # (N, H, W)
    return up / (up.amax(dim=(1, 2), keepdim=True) + 1e-8)          # eps: the map can be all zero

In [ ]:
# Self-checks on a tiny random CNN (no downloads)
torch.manual_seed(0)
features = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(8, 16, 3, padding=1), nn.ReLU())
fc = nn.Linear(16, 5)
tiny = nn.Sequential(features, nn.AdaptiveAvgPool2d(1), nn.Flatten(), fc).eval()
x = torch.randn(2, 3, 16, 12)

cam = grad_cam(tiny, features, x, target_class=0)
assert cam.shape == (2, 8, 6), f"expected (N, h, w) = (2, 8, 6), got {tuple(cam.shape)}"
assert (cam >= 0).all(), "the ReLU keeps only positive evidence"
assert (cam.amax(dim=(1, 2)) > 0).all(), "class 0 has positive evidence in both images"
assert not features._forward_hooks, "remove the forward hook before returning"

# GAP + linear after the target layer: d y^c / d A^k_ij = W[c, k] / (h*w) everywhere,
# so Grad-CAM reduces to CAM: ReLU(sum_k W[c, k] A^k) / (h*w)
with torch.no_grad():
    A = features(x)
    expected = torch.relu(torch.einsum("k,nkij->nij", fc.weight[0], A)) / (8 * 6)
assert torch.allclose(cam, expected, atol=1e-6), "Grad-CAM should equal CAM for a GAP + linear head"

# Frozen model, called from inference code
for p in tiny.parameters():
    p.requires_grad_(False)
with torch.no_grad():
    assert torch.allclose(grad_cam(tiny, features, x, target_class=0), expected, atol=1e-6)

heat = upsample_cam(cam, (16, 12))
assert heat.shape == (2, 16, 12), f"expected (2, 16, 12), got {tuple(heat.shape)}"
assert torch.allclose(heat.amax(dim=(1, 2)), torch.ones(2), atol=1e-5), "each map is scaled to max 1"
assert torch.equal(upsample_cam(torch.zeros(1, 4, 4), (8, 8)), torch.zeros(1, 8, 8)), "an all-zero map stays zero"
print("All self-checks passed")

In [ ]:
# Demo: explain a pretrained ResNet-18 (downloads the weights and a sample image)
import io
import urllib.request

import matplotlib.pyplot as plt
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

weights_enum = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights_enum).eval()

# A real photo makes the heatmap meaningful; falls back to random noise when offline.
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
try:
    with urllib.request.urlopen(url, timeout=10) as f:
        image = Image.open(io.BytesIO(f.read())).convert("RGB")
except Exception as e:
    print(f"Could not download the sample image ({e}); using random noise instead")
    image, _ = datasets.FakeData(size=1, image_size=(3, 224, 224))[0]  # a PIL image

# Preprocess the image for the model (ImageNet statistics)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(image).unsqueeze(0)

In [ ]:
# Grad-CAM of the predicted class on the last residual block (7x7 maps), overlaid on the image
with torch.no_grad():
    predicted_class = model(input_tensor).argmax(dim=1).item()
cam = grad_cam(model, model.layer4[-1], input_tensor, predicted_class)   # (1, 7, 7)
heatmap = upsample_cam(cam, (image.size[1], image.size[0]))[0]          # PIL size is (width, height)

class_name = weights_enum.meta["categories"][predicted_class]
plt.imshow(image)
plt.imshow(heatmap.numpy(), alpha=0.5, cmap="jet")
plt.title(f"Predicted class: {class_name} ({predicted_class})")
plt.axis("off")
plt.show()

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("xai", grad_cam=grad_cam, upsample_cam=upsample_cam)

### Common bugs
- **Backpropagating the softmax probability instead of the logit.** $\partial p_c/\partial A = p_c\big(\partial z_c/\partial A - \sum_j p_j\, \partial z_j/\partial A\big)$: the other classes leak into the map, and the gradient shrinks toward 0 as $p_c \to 1$, so confident predictions give near-empty maps.
- **Averaging the gradients over the batch.** $\alpha^c$ is per sample: average over the spatial dimensions only.
- **Detaching $A$ in the forward hook** (`store["A"] = output.detach()`): $A$ is then no longer in the graph, and the gradient w.r.t. it cannot be computed.
- **No graph at all.** Under `torch.no_grad()`, or on a frozen model whose input does not require grad, $A$ has no `grad_fn`. Use `torch.enable_grad()` and an input that requires grad, as above.
- **Leaking hooks.** A hook that is not removed (or not removed when an exception is raised) keeps capturing tensors on every later forward pass.
- **Train mode.** Dropout and BatchNorm batch statistics change the forward pass, so maps change from call to call. Explain a model in `eval()` mode.
- **Normalising with one maximum for the whole batch**, or before upsampling: each heatmap is scaled by its own maximum after interpolation.

### Interview follow-ups
1. **Why the last convolutional layer?** Deeper layers are more class-specific; earlier layers keep more spatial detail but are less class-discriminative. The Grad-CAM paper argues the last convolutional layer is the best compromise, and maps from early layers are noisy.
2. **Why the ReLU?** We want the features that *increase* the class score; the paper notes that negative regions likely belong to other categories, and that without the ReLU the maps sometimes highlight more than the object.
3. **How is it related to CAM?** With global average pooling followed by a linear layer, Grad-CAM equals CAM divided by $h \cdot w$ (the self-check above). Grad-CAM needs no architecture change or retraining and works for any differentiable head, e.g. captioning or VQA.
4. **The map is only 7×7. How do you get finer explanations?** Guided Grad-CAM multiplies the upsampled map with Guided Backpropagation's pixel-space gradients; a higher-resolution layer gives more detail at the cost of class specificity; Grad-CAM++ (Chattopadhay et al., 2018) weights positive gradients to localise several instances better; Score-CAM (Wang et al., 2020) avoids gradients and weights each activation map by how much masking the input with it raises the class score.
5. **How do you know the explanation reflects the model?** Sanity checks (Adebayo et al., 2018): randomise the weights layer by layer, or train on random labels, and see whether the map changes. Plain gradients and Grad-CAM change a lot (they pass); Guided Backprop and Guided Grad-CAM stay visually similar (they fail), so they partly show edges in the image rather than what the model uses. Also: deletion/insertion curves and the pointing game.
6. **What about ViTs or CLIP?** Take a late block's patch tokens, drop the CLS/register tokens, reshape them to the $h \times w$ grid and apply the same recipe (a common choice is the input of the last block's first LayerNorm). Alternatives are attention rollout (Abnar & Zuidema, 2020) and gradient-weighted relevance propagation (Chefer et al., 2021); raw attention maps alone are not faithful explanations.